# Per component viz of matmus

In [1]:
import os
# Append the standard macOS TeX path to the environment
os.environ["PATH"] += os.pathsep + "/Library/TeX/texbin"

import manim

In [2]:
import mlx.core as mx
import mlx.nn as nn
from mlx.utils import tree_flatten
from mlx_lm.models.qwen3 import Model, ModelArgs

# 1. Define the 122-param Model arguments
model_args = ModelArgs(
    model_type="qwen3", hidden_size=3, num_hidden_layers=1,
    intermediate_size=3, num_attention_heads=1, rms_norm_eps=1e-6,
    vocab_size=10, tie_word_embeddings=True, num_key_value_heads=1,
    max_position_embeddings=64, rope_theta=3, head_dim=4,
)

# 2. Instantiate and load weights
model = Model(model_args)
n_params = sum(p.size for _, p in tree_flatten(model.parameters()))
checkpoint = f"checkpoint/best_{n_params}.npz"

model.load_weights(list(mx.load(checkpoint).items()))
model.eval()
mx.eval(model.parameters())
print(f"Loaded {checkpoint} ({n_params} params)\n")

# 3. Map names to instance IDs so we can track them
TARGET_MODULE_REF = {}
NAME_TO_ID = {}
for name, module in model.named_modules():
    if hasattr(module, "weight"):
        TARGET_MODULE_REF[name] = module
        NAME_TO_ID[name] = id(module)

# 4. Patch the nn.Linear class __call__ directly
SAVED_INPUTS = {}
orig_linear_call = nn.Linear.__call__

def hooked_linear_call(self, x, *args, **kwargs):
    # Save the input using the instance's unique ID
    SAVED_INPUTS[id(self)] = x
    return orig_linear_call(self, x, *args, **kwargs)

nn.Linear.__call__ = hooked_linear_call

# 5. Run a forward pass to trigger the hooks
dummy_seq = [0, 2, 1, 0, 0, 4, 3, 0] 
x = mx.array([dummy_seq], dtype=mx.int32)
_ = model(x) 

print(f"Forward pass complete!")
print("Available linear modules to visualize:")
for name, module_id in NAME_TO_ID.items():
    if module_id in SAVED_INPUTS:
        print(f" - {name}")

# --- SETUP FOR VISUALIZATION ---
TARGET_LAYER_NAME = "model.layers.0.self_attn.q_proj"
TOKEN_IDX = -1

/Users/s2011847/repos/minimal-ten-digit-addition-transformer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded checkpoint/best_122.npz (122 params)

Forward pass complete!
Available linear modules to visualize:
 - model.layers.0.mlp.up_proj
 - model.layers.0.mlp.down_proj
 - model.layers.0.mlp.gate_proj
 - model.layers.0.self_attn.o_proj
 - model.layers.0.self_attn.v_proj
 - model.layers.0.self_attn.k_proj
 - model.layers.0.self_attn.q_proj


In [3]:
NAME_TO_ID

{'model.norm': 5232872016,
 'model.layers.0.post_attention_layernorm': 5232871936,
 'model.layers.0.input_layernorm': 5232871856,
 'model.layers.0.mlp.up_proj': 5232871776,
 'model.layers.0.mlp.down_proj': 5232871696,
 'model.layers.0.mlp.gate_proj': 5232871536,
 'model.layers.0.self_attn.k_norm': 5232870896,
 'model.layers.0.self_attn.q_norm': 5232870576,
 'model.layers.0.self_attn.o_proj': 5232870416,
 'model.layers.0.self_attn.v_proj': 5232870256,
 'model.layers.0.self_attn.k_proj': 5232869456,
 'model.layers.0.self_attn.q_proj': 5232869856,
 'model.embed_tokens': 5232869616}

# now viz!

In [4]:
from manim import *
import numpy as np

# v2

In [ ]:
%%manim -v WARNING -ql VisualizeQwen3MatMul

from manim import *
import numpy as np


class VisualizeQwen3MatMul(ThreeDScene):
    def construct(self):
        # ===== camera =====
        self.set_camera_orientation(phi=70 * DEGREES, theta=-35 * DEGREES, zoom=1.25)

        # ===== fetch W and x from the loaded model =====
        target = TARGET_MODULE_REF[TARGET_LAYER_NAME]
        W_full = np.array(target.weight)                                              # (out, in)
        x_full = np.array(SAVED_INPUTS[NAME_TO_ID[TARGET_LAYER_NAME]][0, TOKEN_IDX])  # (in,)
        token_id = int(dummy_seq[TOKEN_IDX])  # actual token id at TOKEN_IDX

        # We need a 3 -> 3 linear map to actually draw in 3-space.
        W = W_full[:3, :3].astype(float)
        x_vec = x_full[:3].astype(float)

        # ===== thin-lined bounding box cube =====
        box_size = 4
        bbox = Cube(
            side_length=box_size,
            fill_opacity=0,
            stroke_width=1,
            stroke_color=GREY_B,
        )

        # ===== yellow axes =====
        half = box_size / 2
        axes = ThreeDAxes(
            x_range=[-half, half, 1],
            y_range=[-half, half, 1],
            z_range=[-half, half, 1],
            x_length=box_size,
            y_length=box_size,
            z_length=box_size,
            axis_config={
                "color": YELLOW,
                "stroke_width": 3.5,
                "tip_width": 0.18,
                "tip_height": 0.22,
                "include_ticks": True,
            },
        )

        # ===== static "ghost" axes that stay in place =====
        ghost_axes = ThreeDAxes(
            x_range=[-half, half, 1],
            y_range=[-half, half, 1],
            z_range=[-half, half, 1],
            x_length=box_size,
            y_length=box_size,
            z_length=box_size,
            axis_config={
                "color": YELLOW,
                "stroke_width": 1,
                "tip_width": 0.10,
                "tip_height": 0.12,
                "include_ticks": True,
                "tick_size": 0.05,
            },
        )

        # ===== basis vectors: x=red, y=green, z=blue (z is up) =====
        arrow_kw = dict(thickness=0.025, height=0.22, base_radius=0.09)
        x_hat = Arrow3D(start=ORIGIN, end=[1, 0, 0], color=RED,   **arrow_kw)
        y_hat = Arrow3D(start=ORIGIN, end=[0, 1, 0], color=GREEN, **arrow_kw)
        z_hat = Arrow3D(start=ORIGIN, end=[0, 0, 1], color=BLUE,  **arrow_kw)

        # ===== the input vector x (PURPLE) =====
        x_arrow = Arrow3D(
            start=ORIGIN,
            end=x_vec,
            color=PURPLE,
            thickness=0.02,
            height=0.18,
            base_radius=0.07,
        )

        # ===== ValueTracker driving the matmul interpolation =====
        alpha = ValueTracker(0.0)

        # Final endpoints under W
        end_xhat_final = W @ np.array([1.0, 0.0, 0.0])
        end_yhat_final = W @ np.array([0.0, 1.0, 0.0])
        end_zhat_final = W @ np.array([0.0, 0.0, 1.0])
        end_xvec_final = W @ x_vec

        # ===== labels (plain Text, billboard toward camera, constant size) =====
        label_x     = Text("x", font_size=28, color=RED,    weight=BOLD)
        label_y     = Text("y", font_size=28, color=GREEN,  weight=BOLD)
        label_z     = Text("z", font_size=28, color=BLUE,   weight=BOLD)
        label_token = Text(f" '{token_id}' ", font_size=24, color=PURPLE, weight=BOLD)

        LABEL_PAD = 0.25
        TOKEN_PAD = 0.30  # token label a touch further out

        def make_label_updater(initial_end, final_end, pad=LABEL_PAD):
            initial_end = np.array(initial_end, dtype=float)
            final_end   = np.array(final_end,   dtype=float)
            def upd(m):
                a = alpha.get_value()
                end = (1 - a) * initial_end + a * final_end
                n = np.linalg.norm(end)
                if n > 1e-6:
                    m.move_to(end + (pad / n) * end)
                else:
                    m.move_to(end)
            return upd

        label_x.add_updater(make_label_updater([1, 0, 0], end_xhat_final))
        label_y.add_updater(make_label_updater([0, 1, 0], end_yhat_final))
        label_z.add_updater(make_label_updater([0, 0, 1], end_zhat_final))
        label_token.add_updater(make_label_updater(x_vec, end_xvec_final, pad=TOKEN_PAD))
        for m in (label_x, label_y, label_z, label_token):
            m.update()


        # ===== title pinned to the top-right of the frame =====
        title = Text("Attention: Query", font_size=32, color=WHITE, weight=BOLD)
        title.to_corner(UR, buff=0.4)

        # ===== build the scene =====
        self.add(bbox, ghost_axes, axes, x_hat, y_hat, z_hat)
        self.add_fixed_orientation_mobjects(label_x, label_y, label_z)
        self.add_fixed_in_frame_mobjects(title)

        self.wait(1.5)
        self.add(x_arrow)
        self.add_fixed_orientation_mobjects(label_token)
        self.wait(0.5)
        self.begin_ambient_camera_rotation(rate=0.4)
        self.wait(3)
        self.stop_ambient_camera_rotation()
        self.wait(1)

        # ===== apply the matrix =====
        # Swap the static basis/x arrows for redrawing ones whose tip cones
        # stay the same physical size throughout the transformation.
        self.remove(x_hat, y_hat, z_hat, x_arrow)

        def redrawing_arrow(initial_end, final_end, color, **kw):
            initial_end = np.array(initial_end, dtype=float)
            final_end   = np.array(final_end,   dtype=float)
            def builder():
                a = alpha.get_value()
                end = (1 - a) * initial_end + a * final_end
                if np.linalg.norm(end) < 1e-6:
                    end = np.array([1e-6, 0.0, 0.0])
                return Arrow3D(start=ORIGIN, end=end, color=color, **kw)
            return always_redraw(builder)

        x_hat_dyn   = redrawing_arrow([1, 0, 0], end_xhat_final, RED,    **arrow_kw)
        y_hat_dyn   = redrawing_arrow([0, 1, 0], end_yhat_final, GREEN,  **arrow_kw)
        z_hat_dyn   = redrawing_arrow([0, 0, 1], end_zhat_final, BLUE,   **arrow_kw)
        x_arrow_dyn = redrawing_arrow(
            x_vec, end_xvec_final, PURPLE,
            thickness=0.02, height=0.18, base_radius=0.07,
        )

        self.add(x_hat_dyn, y_hat_dyn, z_hat_dyn, x_arrow_dyn)

        # Axes get the real matrix transform; arrows + labels are driven by alpha in lockstep
        self.play(
            ApplyMatrix(W, axes),
            alpha.animate.set_value(1.0),
            run_time=4,
        )

        self.begin_ambient_camera_rotation(rate=0.4)
        self.wait(3)
        self.stop_ambient_camera_rotation()

Manim Community v0.20.1

# what gemini 3.1 pro shat out

In [29]:
from manim import *
import numpy as np

# Inject the extracted data from the Jupyter namespace (modified to use new colors)
def extract_attention_vectors_modified(dummy_seq=[3, 0, 4, 9]):
    """
    Runs a forward pass on 4 tokens and returns a dictionary 
    of their sliced (3D) embeddings, Queries, and Keys, with modified colors.
    """
    # 1. Run the forward pass to populate SAVED_INPUTS
    x_input = mx.array([dummy_seq], dtype=mx.int32)
    _ = model(x_input)
    
    # 2. Get the specific layer instances
    q_name = "model.layers.0.self_attn.q_proj"
    k_name = "model.layers.0.self_attn.k_proj"
    
    # 3. Extract and slice weights for 3D -> 3D transform (top 3 rows/cols)
    W_q_full = np.array(TARGET_MODULE_REF[q_name].weight)
    W_k_full = np.array(TARGET_MODULE_REF[k_name].weight)
    
    W_q = W_q_full[:3, :3].astype(float)
    W_k = W_k_full[:3, :3].astype(float)
    
    # 4. Extract the inputs (embeddings) that went into the linear layers
    # Shape of SAVED_INPUTS for this layer: (batch=1, seq_len=4, hidden=3)
    layer_inputs = np.array(SAVED_INPUTS[NAME_TO_ID[q_name]][0])
    
    # 5. Package it into a dictionary for Manim with modified colors
    vector_dict = {}
    # New distinct colors, avoiding RGB (RED, GREEN, BLUE)
    colors = [ORANGE, PURPLE, TEAL, PINK] 
    
    for i, token_id in enumerate(dummy_seq):
        # Sliced embedding (3D)
        emb = layer_inputs[i, :3].astype(float)
        
        # Calculate the sliced projections manually for verification/easy access
        q_vec = W_q @ emb
        k_vec = W_k @ emb
        
        vector_dict[i] = {
            "token_id": int(token_id),
            "color": colors[i % len(colors)],
            "emb": emb,
            "q": q_vec,
            "k": k_vec,
        }
        
    return vector_dict, W_q, W_k

VDATA, WQ, WK = extract_attention_vectors_modified([3, 0, 4, 9])
TARGET = 3 # The '=' token

In [42]:
# --- HELPER & BASE SCENE UPDATES ---
# (Run this cell to update the colors and the label formatting)

import numpy as np
from manim import *

def extract_attention_vectors_modified(dummy_seq=[3, 0, 4, 9]):
    x_input = mx.array([dummy_seq], dtype=mx.int32)
    _ = model(x_input)
    
    q_name = "model.layers.0.self_attn.q_proj"
    k_name = "model.layers.0.self_attn.k_proj"
    
    W_q_full = np.array(TARGET_MODULE_REF[q_name].weight)
    W_k_full = np.array(TARGET_MODULE_REF[k_name].weight)
    
    W_q = W_q_full[:3, :3].astype(float)
    W_k = W_k_full[:3, :3].astype(float)
    
    layer_inputs = np.array(SAVED_INPUTS[NAME_TO_ID[q_name]][0])
    
    vector_dict = {}
    colors = [ORANGE, PURPLE, TEAL, PINK]  # Replaced MAGENTA with PINK
    
    for i, token_id in enumerate(dummy_seq):
        emb = layer_inputs[i, :3].astype(float)
        q_vec = W_q @ emb
        k_vec = W_k @ emb
        
        vector_dict[i] = {
            "token_id": int(token_id),
            "color": colors[i % len(colors)],
            "emb": emb,
            "q": q_vec,
            "k": k_vec,
        }
        
    return vector_dict, W_q, W_k

VDATA, WQ, WK = extract_attention_vectors_modified([3, 0, 4, 9])
TARGET = 3 

class Qwen3BaseScene(ThreeDScene):
    def setup_scene(self, title_text):
        self.set_camera_orientation(phi=70 * DEGREES, theta=-35 * DEGREES, zoom=1.25)
        
        box_size = 4
        self.bbox = Cube(side_length=box_size, fill_opacity=0, stroke_width=1, stroke_color=GREY_B)
        
        self.axes = ThreeDAxes(
            x_range=[-box_size/2, box_size/2, 1], y_range=[-box_size/2, box_size/2, 1], z_range=[-box_size/2, box_size/2, 1],
            x_length=box_size, y_length=box_size, z_length=box_size,
            axis_config={"color": YELLOW, "stroke_width": 3.5, "tip_width": 0.18, "tip_height": 0.22}
        )
        
        self.ghost_axes = ThreeDAxes(
            x_range=[-box_size/2, box_size/2, 1], y_range=[-box_size/2, box_size/2, 1], z_range=[-box_size/2, box_size/2, 1],
            x_length=box_size, y_length=box_size, z_length=box_size,
            axis_config={"color": YELLOW, "stroke_width": 1, "tip_width": 0.10, "tip_height": 0.12, "tick_size": 0.05}
        )

        arrow_kw = dict(thickness=0.025, height=0.22, base_radius=0.09)
        self.x_basis = Arrow3D(start=ORIGIN, end=[1, 0, 0], color=RED, **arrow_kw)
        self.y_basis = Arrow3D(start=ORIGIN, end=[0, 1, 0], color=GREEN, **arrow_kw)
        self.z_basis = Arrow3D(start=ORIGIN, end=[0, 0, 1], color=BLUE, **arrow_kw)
        
        self.label_x = Text("x", font_size=28, color=RED, weight=BOLD).move_to([1.25, 0, 0])
        self.label_y = Text("y", font_size=28, color=GREEN, weight=BOLD).move_to([0, 1.25, 0])
        self.label_z = Text("z", font_size=28, color=BLUE, weight=BOLD).move_to([0, 0, 1.25])
        
        self.add(self.bbox, self.ghost_axes, self.axes, self.x_basis, self.y_basis, self.z_basis)
        self.add_fixed_orientation_mobjects(self.label_x, self.label_y, self.label_z)

        title = Text(title_text, font_size=32, color=WHITE, weight=BOLD)
        title.to_corner(UR, buff=0.4)
        self.add_fixed_in_frame_mobjects(title)

    def create_vector(self, vec, color, dashed=False, emphasize=False):
        if dashed:
            return DashedLine(start=ORIGIN, end=vec, color=color, dash_length=0.1, stroke_width=4).add_tip(tip_width=0.15, tip_length=0.2)
        else:
            thickness = 0.04 if emphasize else 0.02
            return Arrow3D(start=ORIGIN, end=vec, color=color, thickness=thickness, height=0.18, base_radius=0.07)

    def create_label(self, vec, text, color):
        lbl = Text(f"{text}", font_size=24, color=color, weight=BOLD)
        n = np.linalg.norm(vec)
        pad = 0.3
        target_pos = vec + (pad / n) * vec if n > 1e-6 else vec
        lbl.move_to(target_pos)
        return lbl

    # --- NEW: Helper factory to keep arrows from skewing during transformations ---
    def get_dynamic_transition_helpers(self, alpha_tracker):
        def make_dynamic_arrow(start_vec, end_vec, color, emphasize=False, dashed=False):
            start_vec = np.array(start_vec, dtype=float)
            end_vec = np.array(end_vec, dtype=float)
            def get_end():
                a = alpha_tracker.get_value()
                val = (1 - a) * start_vec + a * end_vec
                if np.linalg.norm(val) < 1e-6:
                    val = np.array([1e-6, 0.0, 0.0])
                return val
            return always_redraw(lambda: self.create_vector(get_end(), color, dashed=dashed, emphasize=emphasize))

        def make_label_updater(start_vec, end_vec):
            start_vec = np.array(start_vec, dtype=float)
            end_vec = np.array(end_vec, dtype=float)
            def updater(m):
                a = alpha_tracker.get_value()
                val = (1 - a) * start_vec + a * end_vec
                n = np.linalg.norm(val)
                if n < 1e-6:
                    m.move_to(val)
                else:
                    m.move_to(val + (0.3 / n) * val)
            return updater
            
        return make_dynamic_arrow, make_label_updater

## Embedding scene

In [ ]:
%%manim -v WARNING -ql ShowEmbeddings

class ShowEmbeddings(Qwen3BaseScene):
    def construct(self):
        self.setup_scene("1. Tokens")
        
        arrows, labels = [], []
        for i, data in VDATA.items():
            arr = self.create_vector(data["emb"], data["color"])
            
            label_text = data["token_id"]
            if label_text == 0:
                label_text = "+"
            elif label_text == 9:
                label_text = "="
                
            lbl = self.create_label(data["emb"], label_text, data["color"])
            arrows.append(arr)
            labels.append(lbl)
            
        self.begin_ambient_camera_rotation(rate=0.3)
        self.wait(2)
            
        self.play(*[GrowFromPoint(arr, ORIGIN) for arr in arrows], run_time=1.5)
        self.add_fixed_orientation_mobjects(*labels)
        
        self.wait(7)
        self.stop_ambient_camera_rotation()

## Query projections

In [ ]:
%%manim -v WARNING -ql ShowQueryProj

class ShowQueryProj(Qwen3BaseScene):
    def construct(self):
        self.setup_scene("2. Queries")
        
        arrows, labels = [], []
        for i, data in VDATA.items():
            arr = self.create_vector(data["emb"], data["color"], emphasize=(i == TARGET))
            
            label_text = data["token_id"]
            if label_text == 0: label_text = "+"
            elif label_text == 9: label_text = "="
                
            lbl = self.create_label(data["emb"], label_text, data["color"])
            arrows.append(arr)
            labels.append(lbl)
            
        self.add(*arrows)
        self.add_fixed_orientation_mobjects(*labels)
        self.wait(1)
        
        # Fade non-targets
        non_target_anims = []
        for i, arr in enumerate(arrows):
            if i != TARGET:
                non_target_anims.append(arr.animate.set_opacity(0.15))
                non_target_anims.append(labels[i].animate.set_opacity(0.15))
        
        self.play(*non_target_anims)
        self.wait(1)
        
        # --- DYNAMIC MATRIX TRANSFORMATION ---
        alpha = ValueTracker(0.0)
        make_arrow, make_updater = self.get_dynamic_transition_helpers(alpha)
        
        # 1. Transform RGB Basis Vectors
        end_x = WQ @ np.array([1.0, 0.0, 0.0])
        end_y = WQ @ np.array([0.0, 1.0, 0.0])
        end_z = WQ @ np.array([0.0, 0.0, 1.0])
        
        dyn_x = make_arrow([1, 0, 0], end_x, RED)
        dyn_y = make_arrow([0, 1, 0], end_y, GREEN)
        dyn_z = make_arrow([0, 0, 1], end_z, BLUE)
        
        self.remove(self.x_basis, self.y_basis, self.z_basis)
        self.add(dyn_x, dyn_y, dyn_z)
        
        self.label_x.add_updater(make_updater([1, 0, 0], end_x))
        self.label_y.add_updater(make_updater([0, 1, 0], end_y))
        self.label_z.add_updater(make_updater([0, 0, 1], end_z))

        # 2. Transform the Target Token Vector
        target_emb = VDATA[TARGET]["emb"]
        target_q = VDATA[TARGET]["q"]
        target_color = VDATA[TARGET]["color"]
        
        dyn_target = make_arrow(target_emb, target_q, target_color, emphasize=True)
        self.remove(arrows[TARGET])
        self.add(dyn_target)
        
        labels[TARGET].add_updater(make_updater(target_emb, target_q))
        
        # Run Animation
        self.play(
            ApplyMatrix(WQ, self.axes),
            alpha.animate.set_value(1.0),
            run_time=3
        )
        
        # Cleanup updaters
        self.label_x.clear_updaters()
        self.label_y.clear_updaters()
        self.label_z.clear_updaters()
        labels[TARGET].clear_updaters()
        
        self.begin_ambient_camera_rotation(rate=0.3)
        self.wait(5)
        self.stop_ambient_camera_rotation()

Manim Community v0.20.1

Waiting 4:  95%|█████████▌| 100/105 [02:55<00:08,  1.74s/it]                                                 

## Key projections

In [ ]:
%%manim -v WARNING -ql ShowKeyProj

class ShowKeyProj(Qwen3BaseScene):
    def construct(self):
        self.setup_scene("3. Keys")
        
        arrows, labels = [], []
        for i, data in VDATA.items():
            arr = self.create_vector(data["emb"], data["color"])
            
            label_text = data["token_id"]
            if label_text == 0: label_text = "+"
            elif label_text == 9: label_text = "="
            
            lbl = self.create_label(data["emb"], label_text, data["color"])
            arrows.append(arr)
            labels.append(lbl)
            
        self.add(*arrows)
        self.add_fixed_orientation_mobjects(*labels)
        self.wait(1)
        
        # --- DYNAMIC MATRIX TRANSFORMATION ---
        alpha = ValueTracker(0.0)
        make_arrow, make_updater = self.get_dynamic_transition_helpers(alpha)
        
        # 1. Transform RGB Basis Vectors
        end_x = WK @ np.array([1.0, 0.0, 0.0])
        end_y = WK @ np.array([0.0, 1.0, 0.0])
        end_z = WK @ np.array([0.0, 0.0, 1.0])
        
        dyn_x = make_arrow([1, 0, 0], end_x, RED)
        dyn_y = make_arrow([0, 1, 0], end_y, GREEN)
        dyn_z = make_arrow([0, 0, 1], end_z, BLUE)
        
        self.remove(self.x_basis, self.y_basis, self.z_basis)
        self.add(dyn_x, dyn_y, dyn_z)
        
        self.label_x.add_updater(make_updater([1, 0, 0], end_x))
        self.label_y.add_updater(make_updater([0, 1, 0], end_y))
        self.label_z.add_updater(make_updater([0, 0, 1], end_z))

        # 2. Transform ALL Token Vectors
        dyn_arrows = []
        for i, arr in enumerate(arrows):
            emb = VDATA[i]["emb"]
            k_vec = VDATA[i]["k"]
            color = VDATA[i]["color"]
            
            dyn_arr = make_arrow(emb, k_vec, color)
            dyn_arrows.append(dyn_arr)
            
            self.remove(arr)
            labels[i].add_updater(make_updater(emb, k_vec))
            
        self.add(*dyn_arrows)
            
        # Run Animation
        self.play(
            ApplyMatrix(WK, self.axes),
            alpha.animate.set_value(1.0),
            run_time=3
        )
        
        # Cleanup updaters
        self.label_x.clear_updaters()
        self.label_y.clear_updaters()
        self.label_z.clear_updaters()
        for lbl in labels:
            lbl.clear_updaters()
            
        self.begin_ambient_camera_rotation(rate=0.3)
        self.wait(7)
        self.stop_ambient_camera_rotation()

Manim Community v0.20.1

## Attention projections

In [41]:
%%manim -v WARNING -ql ShowAttention

class ShowAttention(Qwen3BaseScene):
    def construct(self):
        self.setup_scene("4. The Attention Dot Product")
        
        q_data = VDATA[TARGET]
        q_arr = self.create_vector(q_data["q"], q_data["color"], emphasize=True)
        q_lbl = self.create_label(q_data["q"], f"Q_=", q_data["color"])
        
        self.add(q_arr)
        self.add_fixed_orientation_mobjects(q_lbl)
        
        k_arrows, k_labels = [], []
        for i, data in VDATA.items():
            k_arr = self.create_vector(data["k"], data["color"], dashed=True)
            
            label_text = data["token_id"]
            if label_text == 0: label_text = "+"
            elif label_text == 9: label_text = "="
                
            k_lbl = self.create_label(data["k"], f"K_{label_text}", data["color"])
            k_arrows.append(k_arr)
            k_labels.append(k_lbl)
            
        self.play(*[Create(k) for k in k_arrows], run_time=1.5)
        self.add_fixed_orientation_mobjects(*k_labels)
        self.wait(1)
        
        arcs = []
        smallest_angle = float('inf')
        best_match_idx = -1
        
        for i, k_arr in enumerate(k_arrows):
            q_vec = q_data["q"]
            k_vec = VDATA[i]["k"]
            
            cos_theta = np.dot(q_vec, k_vec) / (np.linalg.norm(q_vec) * np.linalg.norm(k_vec) + 1e-9)
            angle_rad = np.arccos(np.clip(cos_theta, -1.0, 1.0))
            
            if angle_rad < smallest_angle:
                smallest_angle = angle_rad
                best_match_idx = i
                
            l1 = Line(ORIGIN, q_vec)
            l2 = Line(ORIGIN, k_vec)
            arc = Angle(l1, l2, radius=0.6, color=WHITE, stroke_width=3)
            arcs.append(arc)
            
        self.play(*[Create(arc) for arc in arcs])
        self.wait(1)
        
        self.play(
            arcs[best_match_idx].animate.set_color(YELLOW).set_stroke(width=6),
            k_arrows[best_match_idx].animate.set_color(YELLOW)
        )
        
        self.begin_ambient_camera_rotation(rate=0.4)
        self.wait(4)

Manim Community v0.20.1

ValueError: Coords must be in the xy-plane.